# Story 2.5 — Three-Model × Two-Cohort Comparison

**Loop 2 deliverable.** Trains LR, GBM (XGBoost), and EBM on both
cohorts (`hris_only` and `hybrid`) using the temporal train/val split,
then evaluates each on the held-out **validation** set.

Metrics reported per model × cohort:
- **AUC-PR** (primary) — invariant to class imbalance
- **AUC-ROC** (secondary) — context, but optimistic under imbalance
- **Precision@10%** — operational: of the top 10% flagged, how many exit?
- **Precision@20%** — operational: of the top 20% flagged, how many exit?
- **Recall@10%** — FLIP-RISK: what fraction of real exits are in the top 10%?
- **Brier** — calibration: mean squared error of predicted probabilities

> **Test set is held out.** This notebook uses validation scores only.
> Final test-set evaluation runs after threshold selection in Epic 3.

## 1 — Setup

In [ ]:
from __future__ import annotations

import warnings

import pandas as pd

from retention import config
from retention.data.load import load_attrition_features
from retention.data.split import temporal_split
from retention.evaluation.metrics import (
    auc_pr,
    auc_roc,
    brier_score,
    format_rung1_caption,
    precision_at_k,
    recall_at_k,
)
from retention.features.catalog import get_label_name
from retention.features.cohorts import extract_X_y, get_cohort_feature_names, split_cohorts
from retention.models.ebm import train_ebm
from retention.models.lr import train_lr
from retention.models.xgb import RetentionModel

warnings.filterwarnings('ignore')
LABEL = get_label_name()   # 'voluntary_exit_label'
print(f'SEED={config.SEED}  label={LABEL}')

## 2 — Data Loading & Temporal Split

In [ ]:
# config.DATA_DIR resolves from the package location (absolute), so the
# load works regardless of cwd — nbconvert runs with cwd=notebooks/.
df = load_attrition_features(snapshot_dir=config.DATA_DIR / 'raw')
print(f'Loaded {len(df):,} rows × {df.shape[1]} cols')

train_df, val_df, test_df = temporal_split(df)
print(
    f'Train: {len(train_df):,} rows  '
    f'Val: {len(val_df):,} rows  '
    f'Test: {len(test_df):,} rows'
)
print(f'Val base rate: {val_df[LABEL].mean():.1%}')

## 3 — Cohort Splitting

Both cohorts use **identical row indices** — only the column set differs.
`hris_only` has 7 HRIS feature columns; `hybrid` adds 3 survey columns.

In [ ]:
train_cohorts = split_cohorts(train_df)
val_cohorts   = split_cohorts(val_df)

for cohort_name, cdf in train_cohorts.items():
    n_features = len(get_cohort_feature_names(cohort_name))
    print(f'  {cohort_name}: train={len(cdf):,} rows, {n_features} features')

## 4 — Train All 6 Models

In [ ]:
# extract_X_y is defined in src/retention/features/cohorts.py
# (not inline here — notebook-defined functions bypass mypy + pytest)
models = {}
for cohort in ('hris_only', 'hybrid'):
    X_tr, y_tr = extract_X_y(train_cohorts[cohort], cohort)

    # LR
    models[('LR', cohort)] = train_lr(X_tr, y_tr, cohort=cohort)

    # GBM (XGBoost)
    gbm = RetentionModel(cohort=cohort)
    gbm.fit(X_tr, y_tr)
    models[('GBM', cohort)] = gbm

    # EBM
    models[('EBM', cohort)] = train_ebm(X_tr, y_tr, cohort=cohort)

    print(f'[{cohort}] LR, GBM, EBM trained.')

## 5 — Evaluate on Validation Set

In [ ]:
rows = []
for (model_name, cohort), pipeline in models.items():
    X_val, y_val = extract_X_y(val_cohorts[cohort], cohort)
    proba = pipeline.predict_proba(X_val)[:, 1]

    rows.append({
        'Model':      model_name,
        'Cohort':     cohort,
        'AUC-PR':     round(auc_pr(y_val, proba), 3),
        'AUC-ROC':    round(auc_roc(y_val, proba), 3),
        'Prec@10%':   round(precision_at_k(y_val, proba, k=0.10), 3),
        'Prec@20%':   round(precision_at_k(y_val, proba, k=0.20), 3),
        'Rec@10%':    round(recall_at_k(y_val, proba, k=0.10), 3),
        'Brier':      round(brier_score(y_val, proba), 3),
    })

results = pd.DataFrame(rows).set_index(['Model', 'Cohort'])
results

## 6 — Comparison Table

Primary metric: **AUC-PR**. No-skill baseline ≈ val base rate.
All values on the **validation** set (test set held out until Epic 3).

In [ ]:
import matplotlib.pyplot as plt

# Styled table for the notebook
styled = (
    results.style
    .highlight_max(subset=['AUC-PR', 'AUC-ROC', 'Prec@10%', 'Prec@20%', 'Rec@10%'],
                   color='#d4edda', axis=0)
    .highlight_min(subset=['Brier'], color='#d4edda', axis=0)
    .format('{:.3f}')
    .set_caption('Loop 2 — Val-Set Comparison (green = best per metric)')
)
styled

In [ ]:
# AUC-PR bar chart — primary metric
fig, ax = plt.subplots(figsize=(8, 4))
results['AUC-PR'].plot(
    kind='bar', ax=ax, color=['#2166ac', '#4dac26', '#d01c8b'] * 2,
    width=0.6, edgecolor='white', linewidth=0.8
)
base_rate = val_df[LABEL].mean()
ax.axhline(base_rate, color='grey', linestyle='--', linewidth=1,
           label=f'No-skill baseline ({base_rate:.1%} base rate)')
ax.set_xlabel('')
ax.set_ylabel('AUC-PR')
ax.set_title('Loop 2 — AUC-PR by Model × Cohort (validation set)')
ax.legend(fontsize=9)
ax.set_ylim(0, 1)
for bar in ax.patches:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 0.01,
            f'{h:.3f}', ha='center', va='bottom', fontsize=8)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
fig_path = config.REPORTS_DIR / 'figures' / 'loop2_comparison_auc_pr.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print('Saved: reports/figures/loop2_comparison_auc_pr.png')

In [ ]:
# Print Rung 1 captioned winner
best_idx = results['AUC-PR'].idxmax()
best_val = results.loc[best_idx, 'AUC-PR']
print(f'Best validation model: {best_idx[0]} / {best_idx[1]}')
print(format_rung1_caption(best_val))
print()
print('Full results table:')
print(results.to_string())

## 7 — Methodology Note

**OHE tradeoff:** LR and GBM share `build_preprocessor()` which
one-hot-encodes categorical columns (`performance_tier`, `gender`).
EBM uses `build_ebm_preprocessor()` (imputation only) and receives
raw string columns for native categorical bin detection.

This means the three models are not on strictly equal preprocessing
footing — an acknowledged tradeoff, not a hidden one. EBM's native
handling is its natural operating mode; applying OHE to EBM would
degrade its interpretability advantage by fragmenting category levels
into disconnected binary features.

Full treatment: `docs/methodology.md → Cross-Model Comparison Methodology`.
> ⬆ **OPUS:** The honest write-up (Story 2.5.6) is the next step —
> invoke `/opus` when ready to draft `docs/methodology.md`.

**Imbalance handling summary:**
| Model | Strategy |
|-------|----------|
| LR    | `class_weight='balanced'` |
| GBM   | `eval_metric='aucpr'` (default); `scale_pos_weight` available |
| EBM   | `compute_sample_weight('balanced')` passed to `fit()` |